# Importing modules and settings

### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

General settings of Scanpy

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

# Reading the Ingest results file

In [ ]:
# Reading the Ingest results file
adata = sc.read_h5ad('./Smed_L78-L47_20250523_Ingest.h5ad')

In [ ]:
adata

In [ ]:
# Add the names of each Leiden resolution to a list
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names = ['leiden_1',
 'leiden_1.5',
 'leiden_2',
 'leiden_2.5',
 'leiden_3']

In [ ]:
adata.obs.columns

# Declaring samples, clustering layer and ingest column

In [ ]:
samp = 'Sample'

In [ ]:
# Select the best Leiden resolution for your dataset
clusteringlayer = 'leiden_2.5'

In [ ]:
ref = 'sero'

In [ ]:
ingest = clusteringlayer + '_ingested_' + ref

In [ ]:
adata.obs.columns

In [ ]:
print(ingest)

# Calculating the H2B/control ratio

In [ ]:
# Print each unique 
samp_values = adata.obs[samp].unique()
print(samp_values)

In [ ]:
# Count the occurrences of each unique value in a specific column of the adata.obs DataFrame.
adata.obs[samp].value_counts()

In [ ]:
# Verifying that the correct data is selected with the correct leiden resolution
print(f"samp: {samp}")
print(f"clusteringlayer: {clusteringlayer}")

In [ ]:
# Filter both GFP1 and GFP2 populations
adata.obs.loc[(adata.obs[samp] == 'GFP_1') | (adata.obs[samp] == 'GFP_2'), clusteringlayer].value_counts()

In [ ]:
# Filter for both H2B_1 and H2B_2 populations
adata.obs.loc[(adata.obs[samp] == 'H2B_1') | (adata.obs[samp] == 'H2B_2'), clusteringlayer].value_counts()

In [ ]:
# Concatanate the 2 series across the columns (axis=1)
counts_gfp_h2b = pd.concat(
    [adata.obs.loc[(adata.obs[samp] == 'GFP_1') | (adata.obs[samp] == 'GFP_2'), clusteringlayer].value_counts().rename('GFP'),
     adata.obs.loc[(adata.obs[samp] == 'H2B_1') | (adata.obs[samp] == 'H2B_2'), clusteringlayer].value_counts().rename('H2B')],
    axis = 1).sort_index() # Sort the resulting df by index to ensure the clusters are in order.

In [ ]:
counts_gfp_h2b

In [ ]:
# Calculate the percentage of cells in each cluster for both GFP and H2B conditions relative to the total number of cells in each condition.
percs_gfp_h2b = counts_gfp_h2b / counts_gfp_h2b.sum() * 100

In [ ]:
percs_gfp_h2b

In [ ]:
# save in adata.uns
adata.uns['GFP_H2B_perc_' + clusteringlayer] = percs_gfp_h2b

In [ ]:
# Calculate the GFP/H2B ratio
ratios = percs_gfp_h2b['H2B'] / percs_gfp_h2b['GFP']
adata.uns['ratio_H2B/GFP_' + clusteringlayer] = ratios.to_dict()

# Sort the ratios
sorted_ratios = ratios.sort_values(ascending=True)
sorted_ratios

In [ ]:
# bar plot of the ratios

li_colours = [] # order the colours according to the order of the sorted clusters
for i in sorted_ratios.index.tolist():
    li_colours.append(adata.uns[ingest + '_colors'][int(i)])

sns.set_style("white")
plt.figure(figsize=(20,5))
plt.bar(
    x      = sorted_ratios.index.tolist(),
    height = sorted_ratios.values,
    color  = li_colours
)
plt.xticks(rotation=90)
plt.axhline(1, color='gray', linestyle='--')
plt.title('Ratio of H2B cells vs control cells per cluster ( %H2B / % GFP )')
plt.ylabel('GFP / H2B')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# list of the clusters depleted in H2B condition (ratio < 1)
li_h2b = list(ratios[ratios < 1].index)

In [ ]:
# visualisation of the clusters depleted in H2B
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color=clusteringlayer, groups = li_h2b,  size = 20, color_map = umap_cmap)

In [ ]:
# provisional identity of the clusters labeled with ingest
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color=ingest,  size = 20, color_map = umap_cmap)

## 2c/4c ratio

In [ ]:
counts_g1_g2 = pd.concat(
    [adata.obs.loc[(adata.obs[samp] == 'G1a') | (adata.obs[samp] == 'G1b'), clusteringlayer].value_counts().rename('G1'),
     adata.obs.loc[(adata.obs[samp] == 'G2'), clusteringlayer].value_counts().rename('G2')],
    axis = 1).sort_index() # Sort the resulting df by index to ensure the clusters are in order.

In [ ]:
counts_g1_g2

In [ ]:
# Calculate the percentage of cells in each cluster for both GFP and H2B conditions relative to the total number of cells in each condition.
percs_g1_g2 = counts_g1_g2 / counts_g1_g2.sum() * 100

In [ ]:
percs_g1_g2

In [ ]:
# save in adata.uns
adata.uns['G1_G2_perc_' + clusteringlayer] = percs_g1_g2

In [ ]:
# Calculate the GFP/H2B ratio
ratios2 = percs_g1_g2['G2'] / percs_g1_g2['G1']
adata.uns['ratio_G1/G2_' + clusteringlayer] = ratios2.to_dict()

# Sort the ratios
sorted_ratios2 = ratios2.sort_values(ascending=False)
sorted_ratios2

In [ ]:
# Bar plot of the ratios

li_colours = [] # order the colours according to the order of the sorted clusters
for i in sorted_ratios2.index.tolist():
    li_colours.append(adata.uns[ingest + '_colors'][int(i)])


plt.figure(figsize=(20,5), facecolor='white')
plt.bar(
    x      = sorted_ratios2.index.tolist(),
    height = sorted_ratios2.values,
    color  = li_colours    
)
plt.xticks(rotation=90)
plt.title('Ratio of G2 vs G1 per cluster')
plt.ylabel('G2 / G1 Ratio')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
#Calculate log transformed ratios and sort
log_ratios = np.log10(ratios2.replace(0, np.nan))  # Avoid log(0) by replacing 0 with NaN
sorted_log_ratios = log_ratios.sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(20,5), facecolor='white')
plt.bar(
    x      = sorted_log_ratios.index.tolist(),
    height = sorted_log_ratios.values,
    color  = li_colours,           
)
plt.xticks(rotation=90)
plt.title('Log₁₀(G2 / G1) per Cluster')
plt.ylabel('log₁₀(G2 / G1 Ratio)')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# list of the clusters enriched in G2 cells
li_g2 = list(ratios2[ratios2 > 2].index)

In [ ]:
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color=clusteringlayer, groups = li_g2,  size = 20, color_map = umap_cmap)

In [ ]:
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='Condition', groups = ['G1', 'G2'],  size = 20, color_map = umap_cmap)

In [ ]:
# provisional identity of the clusters inferred from ingest
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color=ingest,  size = 20, color_map = umap_cmap)

## 'Neoblasts score'

In [ ]:
# Upload the Wilcox markers into a df
markers_w = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+clusteringlayer]['names']).head(50)

In [ ]:
markers_w

In [ ]:
# Expression of Piwi
with plt.rc_context({'figure.figsize': (8, 8)}):
    sc.pl.umap(adata, color = 'h1SMcG0013999', size = 15, color_map = umap_cmap)

In [ ]:
# main neoblasts clusters are 0 and 1
cl_l = ['0', '1']

In [ ]:
with plt.rc_context({'figure.figsize': (8, 8)}):
    sc.pl.umap(adata, color = clusteringlayer, groups = cl_l, size = 5)

In [ ]:
li_neo_v2 = list(set(markers_w['0'].head(50).to_list() + markers_w['1'].head(50).to_list()))

In [ ]:
len(li_neo_v2)

In [ ]:
# Visualise the markers with the genes based on wilxon method
adata.raw.var.loc[li_neo_v2][['gene_type','gene_ddv6', 'gene_JakkeGuo', 'gene_Jakke_ver1', 'Preferred_name','Description.x']]

In [ ]:
# UMAP of these genes, visualising the expression patterns of these markers across all cells
sc.pl.umap(adata, color=li_neo_v2, cmap = umap_cmap)

In [ ]:
# Dot plot to visualise the expression of the genes
sc.pl.dotplot(adata, li_neo_v2, groupby= clusteringlayer, swap_axes = True, cmap = umap_cmap)

In [ ]:
sc.tl.score_genes(adata, li_neo_v2, score_name = 'neoblast_score')

In [ ]:
# UMAP of the n_genes_by_counts
with plt.rc_context({'figure.figsize': (12, 12)}):
    sc.pl.umap(adata, color='neoblast_score', legend_loc='on data', legend_fontoutline = 5, title= 'neoblast_score', size = 30,
        frameon=False, add_outline = True, color_map = 'viridis')

In [ ]:
# mean neoblast score by cluster
score_cluster = adata.obs.groupby(clusteringlayer)['neoblast_score'].mean('mean').sort_values(ascending = False)

In [ ]:
score_cluster

In [ ]:
# Bar plot of the neoblast score

li_colours = [] # order the colours according to the order of the sorted clusters
for i in score_cluster.index.tolist():
    li_colours.append(adata.uns[ingest + '_colors'][int(i)])


plt.figure(figsize=(20,5), facecolor='white')
plt.bar(
    x      = score_cluster.index.tolist(),
    height = score_cluster.values,
    color  = li_colours    
)
plt.xticks(rotation=90)
plt.title('Neoblast score per cluster')
plt.ylabel('Neoblast score')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()



In [ ]:
#store in adata.uns
adata.uns['neoblast_score_' + clusteringlayer ] = score_cluster.to_dict()

In [ ]:
# cluster 6 is labeled as epidermal cluster
# get the rank of the epidermal cluster in the neoblast score
score_cluster.rank(ascending=False).loc['6']
epi_prog = int(score_cluster.rank(ascending=False).loc['6'])

In [ ]:
# get the list of the clusters with a neoblast score higher than the neoblast cluster
li_neo = list(score_cluster[0:epi_prog].index)

In [ ]:
with plt.rc_context({'figure.figsize': (6, 6)}):
    sc.pl.umap(adata, color=clusteringlayer, groups = li_neo, size = 10)

In [ ]:
# provisional identity of the clusters base on ingest
with plt.rc_context({'figure.figsize': (6, 6)}):
    sc.pl.umap(adata, color=ingest, size = 10)

In [ ]:
# list of the clusters with a neoblast score lower than the epidermal progenitors
li_prog = list(score_cluster[epi_prog:(epi_prog +23 )].index)

In [ ]:
with plt.rc_context({'figure.figsize': (6, 6)}):
    sc.pl.umap(adata, color=clusteringlayer, groups = li_prog, size = 10)

In [ ]:
with plt.rc_context({'figure.figsize': (20, 4)}):
    sc.pl.dendrogram(adata, ingest)

## Calculate Average Doublet Score

In [ ]:
#Check that necessary columns exist
'doublet_score' in adata.obs.columns  # Should be True

In [ ]:
'leiden_2.5' in adata.obs.columns    

In [ ]:
#Calculate average doublet score per cluster
avg_doublet_score = adata.obs.groupby('leiden_2.5')['doublet_score'].mean().sort_values(ascending=False)

In [ ]:
avg_doublet_score

In [ ]:
# Bar plot of the doublet score

li_colours = [] # order the colours according to the order of the sorted clusters
for i in avg_doublet_score.index.tolist():
    li_colours.append(adata.uns[ingest + '_colors'][int(i)])


plt.figure(figsize=(20,5), facecolor='white')
plt.bar(
    x      = avg_doublet_score.index.tolist(),
    height = avg_doublet_score.values,
    color  = li_colours    
)
plt.xticks(rotation=90)
plt.title('Average Doublet Score per Cluster')
plt.ylabel('Doublet score')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# save in adata.uns
adata.uns['doublet_score_' + clusteringlayer ] = avg_doublet_score.to_dict()

# average of n_genes per cluster

In [ ]:
avg_n_genes = adata.obs.groupby('leiden_2.5')['n_genes'].mean().sort_values(ascending=False)

In [ ]:
avg_n_genes

In [ ]:
# Bar plot of the average n_genes

li_colours = [] # order the colours according to the order of the sorted clusters
for i in avg_n_genes.index.tolist():
    li_colours.append(adata.uns[ingest + '_colors'][int(i)])


plt.figure(figsize=(20,5), facecolor='white')
plt.bar(
    x      = avg_n_genes.index.tolist(),
    height = avg_n_genes.values,
    color  = li_colours    
)
plt.xticks(rotation=90)
plt.title('Average n_genes per Cluster')
plt.ylabel('n_genes')
plt.xlabel('Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# save in adata.uns
adata.uns['n_genes_' + clusteringlayer ] = avg_n_genes.to_dict()

# Visualisation

In [ ]:
adata.uns[clusteringlayer + '_colors']

In [ ]:
# Combine everything into one DataFrame
# the variables are series sorted by the cluster numbers
score = pd.DataFrame({
    'H2B_GFP': ratios,
    'G2_G1': ratios2,
    'neoblast_score':  score_cluster.sort_index() , 
    'doublet_score': avg_doublet_score.sort_index(), 
    'n_genes': avg_n_genes.sort_index() ,
    'cluster_size': adata.obs.groupby('leiden_2.5').count()['Experiment'],  # total cells per cluster
    'colours': adata.uns[clusteringlayer + '_colors'] # colours of the clusters
}).dropna()

In [ ]:
score

In [ ]:
# Neoblast score and G1 G2 ratio
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    x=score['neoblast_score'],
    y=score['G2_G1'],
    s=score['cluster_size']/2,  # bubble size
    alpha=0.6,
    color=score['colours'],
    edgecolor='k'
)

plt.xlabel('Neoblast Score')
plt.ylabel('G2 / G1 Ratio')
plt.title('Correlation between G2/G1 ratio and Neoblast Score')
# Annotate cluster IDs
for idx, row in score.iterrows():
    plt.text(row['neoblast_score'], row['G2_G1'], str(idx), fontsize=8, ha='center')

In [ ]:
# log transformed neoblast score and G1 G2 ratio
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    x=np.log10(score['neoblast_score'].replace(0, np.nan)),
    y=np.log10(score['G2_G1'].replace(0, np.nan)),
    s=score['cluster_size']/2,  # bubble size
    alpha=0.6,
    color=score['colours'],
    edgecolor='k'
)

plt.xlabel('Log10 Neoblast Score')
plt.ylabel('Log10 G2 / G1 Ratio')
plt.title('Correlation between log10 G2/G1 ratio and log10 Neoblast Score')
# Annotate cluster IDs
for idx, row in score.iterrows():
    plt.text(np.log10(row['neoblast_score']), np.log10(row['G2_G1']), str(idx), fontsize=8, ha='center')

In [ ]:
# Neoblast score and H2B GFP ratio
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    x=score['neoblast_score'],
    y=score['H2B_GFP'],
    s=score['cluster_size']/2,  # bubble size
    alpha=0.6,
    color=score['colours'],
    edgecolor='k'
)

plt.xlabel('Neoblast Score')
plt.ylabel('H2B / GFP Ratio')
plt.title('Correlation between H2B/GFP ratio and Neoblast Score')
# Annotate cluster IDs
for idx, row in score.iterrows():
    plt.text(row['neoblast_score'], row['H2B_GFP'], str(idx), fontsize=8, ha='center')

In [ ]:
# G1 G2 and H2B GFP ratio
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    x=score['G2_G1'],
    y=score['H2B_GFP'],
    s=score['cluster_size']/2,  # bubble size
    alpha=0.6,
    color=score['colours'],
    edgecolor='k'
)

plt.xlabel('G1/G2 ratio')
plt.ylabel('H2B/GFP ratio')
plt.title('Correlation between G1/G2 ratio and H2B/GFP ratio')
# Annotate cluster IDs
for idx, row in score.iterrows():
    plt.text(row['G2_G1'], row['H2B_GFP'], str(idx), fontsize=8, ha='center')

In [ ]:
# log transformed G1 G2 ratio and H2B GFP ratio
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    x=np.log10(score['G2_G1'].replace(0, np.nan)),
    y=np.log10(score['H2B_GFP'].replace(0, np.nan)),
    s=score['cluster_size']/2,  # bubble size
    alpha=0.6,
    color=score['colours'],
    edgecolor='k'
)

plt.xlabel('Log10 G2 / G1 ratio')
plt.ylabel('Log10 H2B / GFP Ratio')
plt.title('Correlation between log10 G2/G1 ratio and log10 H2B/GFP ratio')
# Annotate cluster IDs
for idx, row in score.iterrows():
    plt.text(np.log10(row['G2_G1']), np.log10(row['H2B_GFP']), str(idx), fontsize=8, ha='center')

In [ ]:
adata.write('./Smed_L78-L47_20250523_Ingest.h5ad')